# Experimental Analysis of Bamboo–Steel Reinforced Concrete Beams
### Recorded flexural loads, reinforcement costs, and multicriteria decision support

**Jeff-Mathieu Saint-Amour**  
Data Analyst · Civil Engineer · PhD Student in Production Engineering

**Analytical environment:** Python · pandas · NumPy · Matplotlib · SQLite · Power BI

## Abstract

This study examines recorded flexural loads and reported reinforcement costs for **39 concrete beam specimens** distributed across three-point bending (**3PT; n = 15**) and four-point bending (**4PT; n = 24**). Eight experimental groups form 13 available test–group combinations, with three specimens per combination. A reproducible workflow combines relational validation, descriptive statistics, same-test reference comparisons, load-to-cost ratios, Pareto analysis, and an explicitly weighted multicriteria scenario.

Under 3PT, VAB-III has the highest group mean load (**80.68 kN**) and the highest bamboo-containing load-to-cost ratio (**4.61 kN/BRL**). Under 4PT, the steel-only VA reference has the highest overall mean load (**80.64 kN**); VB-III leads bamboo-containing candidates in mean load (**75.79 kN**), while VBC-II leads their load-to-cost ratio (**5.21 kN/BRL**). The equal-weight normalized scenario ranks VAB-II first in 3PT and VBC-II first in 4PT. These outcomes demonstrate the dependence of candidate selection on the stated criterion. Inference remains descriptive and specific to the recorded experiment.

## Data Source

The original dataset is organized in an Excel workbook containing test setups, experimental groups, specimen characteristics, recorded flexural results, reported costs, and a summary worksheet. The analytical workflow also accepts specimen and reference tables exported from the Power BI model. These research inputs are not distributed in this public repository.

Each specimen is linked to one recorded load and one reported reinforcement cost. Input values and measurement units are preserved. File fingerprints document the active source. When CSV inputs are used, the original Excel summary worksheet is unavailable and its independent reconciliation is explicitly marked as not executed.

## Research objectives

1. Establish the integrity and traceability of specimen-level measurements and experimental identifiers.
2. Characterize recorded load and dispersion within each available test–group combination.
3. Quantify same-test differences relative to the steel-only VA and unreinforced VCS references.
4. Evaluate reported reinforcement cost and load-to-cost trade-offs among candidate configurations.
5. Assess the sensitivity of normalized candidate rankings to explicit load and cost weights.
6. Reconcile independent statistical calculations with the relational SQL implementation.

## Analytical scope

The unit of analysis is one specimen. Test setups define separate comparison populations. The dataset has equal specimen counts within available combinations but does not contain every group under both setups. Recorded loads are expressed in kN, dimensions in mm, and reported reinforcement costs in BRL. The workflow supports experimental interpretation and decision exploration; structural design verification requires additional protocol and material information.

## Collaborative research and data availability

This project was developed through collaborative research. This portfolio presents the analytical workflow, methodology, and selected aggregate results. It does not claim sole authorship or ownership of the experimental dataset.

Specimen-level observations and source files are not publicly distributed while the associated scientific work is being prepared. Access may be considered upon reasonable request, subject to prior agreement among the research collaborators and relevant data custodians, applicable institutional requirements, and publication arrangements. Data release is not automatic.

Execution of the notebook requires authorized input data. Saved cell outputs and specimen-level figures have been removed from this public version.

## 1. Input configuration and source inventory

**Objective.** Resolve the input source and document its structure.

The primary input is `Bamboo_Beam_Experimental_Data_Jeff.xlsx`, optionally located through `BAMBOO_INPUT_FILE`. Its six worksheets preserve the experimental records and supplied summary. The alternative input comprises `fact_beam_analysis.csv`, `test_setup.csv`, and `reinforcement_groups.csv`, with an optional directory override through `BAMBOO_DATA_DIR`.

CSV mode reconstructs normalized analytical tables from the specimen records. The original Excel summary is unavailable in that mode, so its reconciliation is explicitly excluded. `BAMBOO_OUTPUT_DIR` controls the export directory. Runtime dependencies are pandas, NumPy, Matplotlib, and openpyxl for Excel ingestion; SQLite is provided by Python.

In [ ]:
from pathlib import Path
import os
import json
import math
import sqlite3
import hashlib
import platform
import statistics
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

INPUT_FILE = Path(os.environ.get("BAMBOO_INPUT_FILE", "Bamboo_Beam_Experimental_Data_Jeff.xlsx"))
OUTPUT_DIR = Path(os.environ.get("BAMBOO_OUTPUT_DIR", "bamboo_outputs"))
CSV_DIR, CHART_DIR = OUTPUT_DIR / "csv", OUTPUT_DIR / "charts"
for folder in [OUTPUT_DIR, CSV_DIR, CHART_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def show_table(table, limit=20):
    if "display" in globals():
        display(table.head(limit))
    else:
        print(table.head(limit).to_string(index=False))

plt.rcParams.update({"figure.dpi": 125, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titlesize": 12})
GROUP_ORDER = ["VCS", "VA", "VAB-II", "VAB-III", "VB-II", "VB-III", "VB-IV", "VBC-II"]
PALETTE = dict(zip(GROUP_ORDER, ["#A0A7AE", "#17324D", "#3086B3", "#56A1C4", "#2C8C67", "#57A97D", "#A7C17A", "#D69A35"]))
def save_chart(fig, filename):
    fig.savefig(CHART_DIR / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)

SHEETS = ["Test_setup", "Reinforcement_Groups", "Specimens", "Flexural_Results", "Specimen_Costs", "Flexural_Results_Summary"]
CSV_NAMES = ["fact_beam_analysis.csv", "test_setup.csv", "reinforcement_groups.csv"]
data_candidates = [Path(os.environ.get("BAMBOO_DATA_DIR", "data")), Path("../data"), INPUT_FILE.parent / "data"]
DATA_DIR = next((folder for folder in data_candidates if all((folder / name).exists() for name in CSV_NAMES)), None)
if not INPUT_FILE.exists() and DATA_DIR is None:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Input workbook or the three analytical CSV input tables are required.")
    uploaded = files.upload()
    excel_candidates = [Path(name) for name in uploaded if name.lower().endswith(".xlsx")]
    if len(excel_candidates) == 1:
        INPUT_FILE = excel_candidates[0]
    elif all(Path(name).exists() for name in CSV_NAMES):
        DATA_DIR = Path(".")
    else:
        raise ValueError("Input requires one Excel workbook or all three named CSV tables.")
SOURCE_SUMMARY_AVAILABLE = INPUT_FILE.exists()
if SOURCE_SUMMARY_AVAILABLE:
    SOURCE_MODE = "excel_workbook"
    SOURCE_PATHS = [INPUT_FILE]
    raw = pd.read_excel(INPUT_FILE, sheet_name=SHEETS)
else:
    SOURCE_MODE = "power_bi_csv_exports"
    SOURCE_PATHS = [DATA_DIR / name for name in CSV_NAMES]
    fact_input = pd.read_csv(SOURCE_PATHS[0], dtype={"specimen_id": "string", "specimen_label": "string", "group_id": "string"})
    raw = {
        "Test_setup": pd.read_csv(SOURCE_PATHS[1]),
        "Reinforcement_Groups": pd.read_csv(SOURCE_PATHS[2], dtype={"group_id": "string"}).drop(columns="is_reference", errors="ignore"),
        "Specimens": fact_input[["specimen_id", "specimen_label", "test_code", "group_name", "test_number", "width_mm", "height_mm", "length_mm", "steel_ratio_pct", "bamboo_ratio_pct"]].rename(columns={"steel_ratio_pct": "steel_ratio_%", "bamboo_ratio_pct": "bamboo_ratio_%"}),
        "Flexural_Results": fact_input[["load_id", "specimen_id", "specimen_label", "flexural_load_kn"]].rename(columns={"load_id": "Flexural load_id", "flexural_load_kn": "Flexural load_kN"}),
        "Specimen_Costs": fact_input[["cost_id", "specimen_id", "specimen_label", "steel_ratio_pct_source", "bamboo_ratio_pct_source", "cost_brl"]].rename(columns={"steel_ratio_pct_source": "steel_ratio", "bamboo_ratio_pct_source": "bamboo_ratio", "cost_brl": "cost_R$"}),
        "Flexural_Results_Summary": pd.DataFrame(columns=["test_code", "group_name", "specimen_count", "mean_Flexural load_kN", "min_Flexural load_kN", "max_Flexural load_kN"])
    }
inventory = pd.DataFrame([{"sheet_or_table": key, "rows": len(value), "columns": len(value.columns),
                           "missing_cells": int(value.isna().sum().sum())} for key, value in raw.items()])
print("Input mode:", SOURCE_MODE)
show_table(inventory)


## 2. Field normalization and measurement preservation

**Objective.** Standardize field names without altering source measurements.

The preparation layer normalizes column labels and trims text fields. Nominal reinforcement ratios remain separate from the higher-precision ratios associated with cost calculations. Ratio values are percentages: `0.68` means **0.68%**. Loads remain in kN, dimensions in mm, and reported reinforcement costs in BRL. Original source files are unchanged.

In [ ]:
tests = raw["Test_setup"].copy()
groups = raw["Reinforcement_Groups"].copy()
groups["is_reference"] = groups.group_name.isin(["VA", "VCS"]).astype(int)
specimens = raw["Specimens"].rename(columns={"steel_ratio_%": "steel_ratio_pct", "bamboo_ratio_%": "bamboo_ratio_pct"}).copy()
results = raw["Flexural_Results"].rename(columns={"Flexural load_id": "load_id", "Flexural load_kN": "flexural_load_kn"}).copy()
costs = raw["Specimen_Costs"].rename(columns={"steel_ratio": "steel_ratio_pct_source",
    "bamboo_ratio": "bamboo_ratio_pct_source", "cost_R$": "cost_brl"}).copy()
summary_source = raw["Flexural_Results_Summary"].rename(columns={
    "mean_Flexural load_kN": "mean_load_kn", "min_Flexural load_kN": "min_load_kn",
    "max_Flexural load_kN": "max_load_kn"}).copy()
for table in [tests, groups, specimens, results, costs, summary_source]:
    table.columns = table.columns.str.strip()
    for col in table.select_dtypes(include="object"):
        table[col] = table[col].astype("string").str.strip()

## 3. Experimental design and data integrity

**Objective.** Validate identifiers, relationships, measurements, and available experimental combinations.

Controls cover primary keys, required values, one-to-one result and cost links, test-code consistency, positive dimensions and loads, nonnegative reinforcement percentages, cross-table labels, and ratio precision. The joined table retains one row per specimen.

The coverage matrix distinguishes unavailable test–group combinations from missing measurements. VCS, VB-IV, and VBC-II occur only in 4PT. VB-II and VBC-II remain separate groups despite equal nominal ratios. Repeated geometry is documented without treating specimens as duplicates or paired observations.

In [ ]:
checks = []
def check(name, condition, detail):
    checks.append({"check": name, "passed": bool(condition), "detail": detail})

for name, table, key in [("test_setup", tests, "test_number"), ("reinforcement_groups", groups, "group_id"),
                          ("specimens", specimens, "specimen_id"), ("flexural_results", results, "load_id"),
                          ("specimen_costs", costs, "cost_id")]:
    check(name+": primary key", table[key].notna().all() and not table[key].duplicated().any(), key)
    check(name+": required values", not table.isna().any().any(), "No automatic imputation")
check("Workbook specimen count", len(specimens)==39, "Expected source inventory: 39")
check("One result per specimen", not results.specimen_id.duplicated().any() and set(results.specimen_id)==set(specimens.specimen_id), "One-to-one result link")
check("One cost per specimen", not costs.specimen_id.duplicated().any() and set(costs.specimen_id)==set(specimens.specimen_id), "One-to-one cost link")
check("Test reference codes", specimens.test_code.isin(tests.test_code).all(), "Known test codes")
check("Group references", specimens.group_name.isin(groups.group_name).all(), "Known reinforcement groups")
mapped = specimens.merge(tests[["test_number", "test_code"]], on="test_number", suffixes=("", "_expected"), validate="many_to_one")
check("Test number/code agreement", mapped.test_code.eq(mapped.test_code_expected).all(), "Redundant source keys must agree")
check("Positive geometry", specimens[["width_mm", "height_mm", "length_mm"]].gt(0).all().all(), "Millimeters")
check("Positive loads and costs", results.flexural_load_kn.gt(0).all() and costs.cost_brl.gt(0).all(), "kN and BRL")
check("Valid nominal reinforcement percentages", specimens[["steel_ratio_pct", "bamboo_ratio_pct"]].ge(0).all().all() and specimens[["steel_ratio_pct", "bamboo_ratio_pct"]].le(100).all().all(), "Stored as percent values, not fractions")
label_audit = specimens[["specimen_id", "specimen_label"]].merge(results[["specimen_id", "specimen_label"]], on="specimen_id", suffixes=("", "_result"), validate="one_to_one").merge(costs[["specimen_id", "specimen_label"]].rename(columns={"specimen_label": "specimen_label_cost"}), on="specimen_id", validate="one_to_one")
check("Specimen labels across sheets", label_audit.specimen_label.eq(label_audit.specimen_label_result).all() and label_audit.specimen_label.eq(label_audit.specimen_label_cost).all(), "Labels retained; IDs used for joins")
ratio_audit = specimens[["specimen_id", "steel_ratio_pct", "bamboo_ratio_pct"]].merge(costs[["specimen_id", "steel_ratio_pct_source", "bamboo_ratio_pct_source"]], on="specimen_id", validate="one_to_one")
ratio_audit["steel_delta_percentage_points"] = ratio_audit.steel_ratio_pct_source-ratio_audit.steel_ratio_pct
ratio_audit["bamboo_delta_percentage_points"] = ratio_audit.bamboo_ratio_pct_source-ratio_audit.bamboo_ratio_pct
check("Steel percentage rounding agreement", ratio_audit.steel_delta_percentage_points.abs().le(0.005).all(), "Tolerance: half a 0.01 percentage-point rounding unit; preserve both precisions")
check("Bamboo percentage agreement", ratio_audit.bamboo_delta_percentage_points.abs().le(1e-9).all(), "No conversion from percent to fraction")
quality_checks = pd.DataFrame(checks)
show_table(quality_checks, 30)
if not quality_checks.passed.all():
    raise ValueError("A validation check failed. Inspect quality_checks before continuing.")

group_map = groups.set_index("group_name").group_id
specimens["group_id"] = specimens.group_name.map(group_map)
specimens_db = specimens[["specimen_id", "specimen_label", "test_number", "group_id", "width_mm", "height_mm", "length_mm", "steel_ratio_pct", "bamboo_ratio_pct"]].copy()
results_db = results[["load_id", "specimen_id", "flexural_load_kn"]].copy()
costs_db = costs[["cost_id", "specimen_id", "steel_ratio_pct_source", "bamboo_ratio_pct_source", "cost_brl"]].copy()
summary_source["group_id"] = summary_source.group_name.map(group_map)
summary_source_db = summary_source[["test_code", "group_id", "specimen_count", "mean_load_kn", "min_load_kn", "max_load_kn"]].copy()
joined = specimens.merge(results_db, on="specimen_id", validate="one_to_one").merge(costs_db, on="specimen_id", validate="one_to_one").merge(groups[["group_id", "is_reference"]], on="group_id", validate="many_to_one")
joined["recorded_load_per_brl"] = joined.flexural_load_kn / joined.cost_brl
check("Join preserves specimen grain", len(joined)==39, "No many-to-many multiplication")
coverage = pd.crosstab(joined.group_name, joined.test_code).reindex(GROUP_ORDER).fillna(0).astype(int)
show_table(coverage.reset_index())
geometry_reuse = specimens.groupby(["width_mm", "height_mm", "length_mm"]).agg(specimens=("specimen_id", "size"), ids=("specimen_id", lambda s: ", ".join(s)), tests=("test_code", lambda s: ", ".join(sorted(set(s))))).reset_index()
geometry_reuse = geometry_reuse[geometry_reuse.specimens>1]

## 4. Descriptive statistics and numerical reconciliation

**Objective.** Characterize the location and dispersion of recorded loads within each test–group combination and verify the numerical implementation.

For recorded loads \(P_1,\ldots,P_n\), the arithmetic mean, sample standard deviation, and coefficient of variation are defined as:

$$
\bar P=\frac{1}{n}\sum_{i=1}^{n}P_i,\qquad
s_P=\sqrt{\frac{\sum_{i=1}^{n}(P_i-\bar P)^2}{n-1}},\qquad
CV_P=100\frac{s_P}{\bar P}.
$$

Each available combination has \(n=3\). Minimum and maximum loads retain the observed range. Mean reported reinforcement cost and the ratio of group means, \(\bar P/\bar C\), provide complementary economic descriptors. The ratio of group means is distinct from the mean of specimen-level ratios when individual costs differ.

Independent calculations with `math.fsum` and `statistics.stdev` check pandas means and sample standard deviations. Excel mode additionally reconciles specimen-level summaries with the original summary worksheet. CSV mode explicitly excludes that unavailable source-summary check.

Sample standard deviation quantifies observed specimen dispersion; it is not a confidence interval or a standard error. No load-to-stress conversion or cross-setup pooling is performed.

In [ ]:
summary = (joined.groupby(["test_code", "group_id", "group_name"], sort=False)
 .agg(n=("flexural_load_kn", "count"), mean_load_kn=("flexural_load_kn", "mean"),
      min_load_kn=("flexural_load_kn", "min"), max_load_kn=("flexural_load_kn", "max"),
      sample_sd_kn=("flexural_load_kn", "std"), mean_cost_brl=("cost_brl", "mean"),
      min_cost_brl=("cost_brl", "min"), max_cost_brl=("cost_brl", "max"))
 .reset_index())
summary["cv_pct"] = 100*summary.sample_sd_kn / summary.mean_load_kn
summary["mean_load_per_mean_cost"] = summary.mean_load_kn / summary.mean_cost_brl
summary["is_reference"] = summary.group_name.isin(["VA", "VCS"]).astype(int)
if SOURCE_SUMMARY_AVAILABLE:
    summary_reconciliation = summary.merge(summary_source_db, on=["test_code", "group_id"], suffixes=("", "_source"), validate="one_to_one", how="outer", indicator=True)
    assert summary_reconciliation._merge.eq("both").all()
    assert summary_reconciliation.n.eq(summary_reconciliation.specimen_count).all()
    for metric in ["mean_load_kn", "min_load_kn", "max_load_kn"]:
        summary_reconciliation[metric+"_difference"] = summary_reconciliation[metric]-summary_reconciliation[metric+"_source"]
        assert np.allclose(summary_reconciliation[metric], summary_reconciliation[metric+"_source"], rtol=0, atol=1e-8)
else:
    summary_reconciliation = pd.DataFrame(columns=["test_code", "group_id", "_merge"])
    print("Excel source-summary reconciliation: not executed (CSV input mode).")
independent_rows=[]
for keys, g in joined.groupby(["test_code", "group_name"]):
    values=g.flexural_load_kn.tolist()
    independent_rows.append({"test_code":keys[0], "group_name":keys[1],
        "mean_independent":math.fsum(values)/len(values), "sd_independent":statistics.stdev(values)})
independent_check=summary.merge(pd.DataFrame(independent_rows),on=["test_code","group_name"],validate="one_to_one")
assert np.allclose(independent_check.mean_load_kn,independent_check.mean_independent,atol=1e-10)
assert np.allclose(independent_check.sample_sd_kn,independent_check.sd_independent,atol=1e-10)
show_table(summary)

## 5. Reference-relative comparisons

**Objective.** Quantify differences relative to available same-test reference groups.

VA denotes steel-only reinforcement, and VCS denotes the unreinforced concrete reference. For group \(g\) and reference \(r\) in the same test setup:

$$
\Delta P_g=\bar P_g-\bar P_r,\qquad
\Delta P_{g,\%}=100\left(\frac{\bar P_g}{\bar P_r}-1\right),\qquad
\Delta C_{g,\%}=100\left(\frac{\bar C_g}{\bar C_r}-1\right).
$$

Positive and negative percentage values retain direction. A positive cost change indicates higher reported reinforcement cost. VCS comparisons are unavailable in 3PT and remain missing rather than being extrapolated.

These descriptive contrasts are conditional on the recorded experimental combinations. They do not establish statistical significance, a causal treatment effect, or structural design suitability.

In [ ]:
reference_comparisons = summary.copy()
for name in ["VA", "VCS"]:
    ref=summary[summary.group_name.eq(name)][["test_code","mean_load_kn","mean_cost_brl"]].rename(columns={"mean_load_kn":f"{name.lower()}_mean_load_kn","mean_cost_brl":f"{name.lower()}_mean_cost_brl"})
    reference_comparisons=reference_comparisons.merge(ref,on="test_code",how="left",validate="many_to_one")
    reference_comparisons[f"load_change_vs_{name.lower()}_pct"]=100*(reference_comparisons.mean_load_kn/reference_comparisons[f"{name.lower()}_mean_load_kn"]-1)
    reference_comparisons[f"cost_change_vs_{name.lower()}_pct"]=100*(reference_comparisons.mean_cost_brl/reference_comparisons[f"{name.lower()}_mean_cost_brl"]-1)
show_table(reference_comparisons[["test_code","group_name","load_change_vs_va_pct","cost_change_vs_va_pct","load_change_vs_vcs_pct"]])

## 6. Multicriteria model, Pareto efficiency, and sensitivity

**Objective.** Evaluate candidate groups under explicit preferences while distinguishing weighted scores from raw load-to-cost ratios.

VA and VCS remain benchmarks and are excluded from the candidate set. For each test setup, candidate mean loads and mean costs receive the following normalized scores:

$$
L_g=\frac{\bar P_g-\min_h\bar P_h}{\max_h\bar P_h-\min_h\bar P_h},\qquad
K_g=\frac{\max_h\bar C_h-\bar C_g}{\max_h\bar C_h-\min_h\bar C_h}.
$$

The weighted score is:

$$
S_g(w)=wL_g+(1-w)K_g,\qquad 0\leq w\leq1.
$$

The baseline uses \(w=0.50\). Sensitivity examines 11 equally spaced weights from 0 to 1. A criterion with zero range receives a score of 1 for every candidate. Ties retain equal rank.

A candidate is Pareto-efficient when no other candidate in the same setup provides at least as much mean load at no greater mean cost, with a strict improvement in at least one criterion. Pareto status is conditional on the stated candidate set and measured criteria.

The equal-weight model is an explicit analytical scenario rather than an elicited stakeholder preference. Raw ratio ranking and normalized weighted ranking use different objective functions. Absolute scores have no common comparison scale across 3PT and 4PT, and candidate-set changes can alter normalization and ranking.

In [ ]:
# Candidate groups exclude controls VA and VCS. No weighting is inferred from data.
DEFAULT_LOAD_WEIGHT=0.50
WEIGHT_GRID=np.round(np.linspace(0,1,11),2)
candidates=summary[summary.is_reference.eq(0)].copy()
normalized=[]
for test,g in candidates.groupby("test_code"):
    g=g.copy()
    load_span=g.mean_load_kn.max()-g.mean_load_kn.min()
    cost_span=g.mean_cost_brl.max()-g.mean_cost_brl.min()
    g["normalized_load"]=1.0 if load_span==0 else (g.mean_load_kn-g.mean_load_kn.min())/load_span
    g["normalized_cost"]=1.0 if cost_span==0 else (g.mean_cost_brl.max()-g.mean_cost_brl)/cost_span
    g["pareto_efficient"]=[int(not (((g.mean_cost_brl<=row.mean_cost_brl)&(g.mean_load_kn>=row.mean_load_kn)&((g.mean_cost_brl<row.mean_cost_brl)|(g.mean_load_kn>row.mean_load_kn))).any())) for _,row in g.iterrows()]
    normalized.append(g)
normalized=pd.concat(normalized,ignore_index=True)
mcda=normalized.copy()
mcda["load_weight"]=DEFAULT_LOAD_WEIGHT
mcda["cost_weight"]=1-DEFAULT_LOAD_WEIGHT
mcda["score"]=mcda.normalized_load*DEFAULT_LOAD_WEIGHT+mcda.normalized_cost*(1-DEFAULT_LOAD_WEIGHT)
mcda["rank"]=mcda.groupby("test_code").score.rank(method="min",ascending=False).astype(int)
sensitivity=pd.concat([normalized.assign(load_weight=w,cost_weight=1-w,score=w*normalized.normalized_load+(1-w)*normalized.normalized_cost) for w in WEIGHT_GRID],ignore_index=True)
sensitivity["rank"]=sensitivity.groupby(["test_code","load_weight"]).score.rank(method="min",ascending=False).astype(int)
show_table(mcda.sort_values(["test_code","rank","group_name"])[["test_code","group_name","mean_load_kn","mean_cost_brl","normalized_load","normalized_cost","score","rank","pareto_efficient"]])

## 7. Analytical visualization

**Objective.** Communicate specimen dispersion, load–cost trade-offs, reference differences, and decision sensitivity.

Four figures present group means with sample standard deviation and specimen points; mean load versus mean cost; percentage load change relative to VA; and weighted-score sensitivity. Test setups remain separated throughout.

Repeated identical cost entries do not provide independent evidence of cost uncertainty. Sensitivity curves describe the effect of stated weights rather than uncertainty in experimental measurements.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.8),sharey=True)
for ax,test in zip(axes,["3PT","4PT"]):
    g=summary[summary.test_code.eq(test)].set_index("group_name").reindex([x for x in GROUP_ORDER if x in set(summary[summary.test_code.eq(test)].group_name)]).reset_index()
    ax.bar(g.group_name,g.mean_load_kn,yerr=g.sample_sd_kn,capsize=4,color=[PALETTE[x] for x in g.group_name],alpha=.75)
    for x,group in enumerate(g.group_name):
        values=joined.loc[joined.test_code.eq(test)&joined.group_name.eq(group),"flexural_load_kn"].to_numpy()
        ax.scatter(x+np.linspace(-.12,.12,len(values)),values,color="#172B3A",s=20,zorder=3)
    ax.set(title=test,xlabel="Reinforcement group")
    ax.tick_params(axis="x",rotation=45)
axes[0].set_ylabel("Recorded flexural load (kN)")
fig.suptitle("Recorded loads by group and test setup — mean ± sample SD; points = specimens")
fig.tight_layout()
save_chart(fig,"01_loads_by_test_and_group.png")

fig,axes=plt.subplots(1,2,figsize=(12,4.8),sharey=True)
for ax,test in zip(axes,["3PT","4PT"]):
    for _,row in summary[summary.test_code.eq(test)].iterrows():
        ax.scatter(row.mean_cost_brl,row.mean_load_kn,color=PALETTE[row.group_name],s=65)
        offset={("4PT","VAB-II"):(6,-13),("4PT","VB-III"):(6,10)}.get((test,row.group_name),(5,5))
        ax.annotate(row.group_name,(row.mean_cost_brl,row.mean_load_kn),xytext=offset,textcoords="offset points",fontsize=8)
    ax.set(title=test,xlabel="Reported mean cost (BRL/specimen)")
axes[0].set_ylabel("Mean recorded flexural load (kN)")
fig.suptitle("Recorded load versus reported cost — descriptive group means")
fig.tight_layout();save_chart(fig,"02_load_cost_tradeoff.png")

fig,axes=plt.subplots(1,2,figsize=(12,4.8))
for ax,test in zip(axes,["3PT","4PT"]):
    g=reference_comparisons[reference_comparisons.test_code.eq(test)&~reference_comparisons.group_name.eq("VA")]
    ax.bar(g.group_name,g.load_change_vs_va_pct,color=[PALETTE[x] for x in g.group_name])
    ax.axhline(0,color="#333333",linewidth=.8)
    ax.set(title=test,xlabel="Group",ylabel="Mean load change vs VA (%)")
    ax.tick_params(axis="x",rotation=45)
fig.suptitle("Comparison with the steel-only reference within each test setup")
fig.tight_layout();save_chart(fig,"03_relative_to_va.png")

fig,axes=plt.subplots(1,2,figsize=(12,4.8))
for ax,test in zip(axes,["3PT","4PT"]):
    for group,g in sensitivity[sensitivity.test_code.eq(test)].groupby("group_name"):
        ax.plot(g.load_weight,g.score,label=group,color=PALETTE[group],marker="o",markersize=3)
    ax.set(title=test,xlabel="Load weight (cost weight = 1 − load weight)",ylabel="Weighted min–max score",ylim=(-.03,1.03))
    ax.legend(fontsize=8,frameon=False)
fig.suptitle("Multicriteria sensitivity — candidates only; normalization within each setup")
fig.tight_layout();save_chart(fig,"04_mcda_weight_sensitivity.png")

## 8. Relational model and analytical SQL views

**Objective.** Persist the specimen records and decision settings in a portable SQLite database.

Normalized tables retain specimens, test setups, experimental groups, recorded loads, reported costs, source-summary records when available, decision settings, sensitivity weights, and project metadata. Analytical views calculate specimen detail, group summaries, reference comparisons, normalized criteria, weighted rankings, Pareto efficiency, and sensitivity results.

The source summary serves reconciliation rather than adding observations. The authoritative baseline load weight is stored in `analysis_settings`. Each execution rebuilds the output database. CSV mode leaves the source-summary table empty and records that provenance explicitly.

In [ ]:
schema_sql="""
CREATE TABLE test_setup(test_number INTEGER PRIMARY KEY,test_code TEXT NOT NULL UNIQUE,test_name TEXT NOT NULL,description TEXT NOT NULL);
CREATE TABLE reinforcement_groups(group_id TEXT PRIMARY KEY,group_name TEXT NOT NULL UNIQUE,description TEXT NOT NULL,is_reference INTEGER NOT NULL CHECK(is_reference IN(0,1)));
CREATE TABLE specimens(specimen_id TEXT PRIMARY KEY,specimen_label TEXT NOT NULL UNIQUE,test_number INTEGER NOT NULL REFERENCES test_setup(test_number),group_id TEXT NOT NULL REFERENCES reinforcement_groups(group_id),width_mm REAL NOT NULL CHECK(width_mm>0),height_mm REAL NOT NULL CHECK(height_mm>0),length_mm REAL NOT NULL CHECK(length_mm>0),steel_ratio_pct REAL NOT NULL CHECK(steel_ratio_pct BETWEEN 0 AND 100),bamboo_ratio_pct REAL NOT NULL CHECK(bamboo_ratio_pct BETWEEN 0 AND 100));
CREATE TABLE flexural_results(load_id INTEGER PRIMARY KEY,specimen_id TEXT NOT NULL UNIQUE REFERENCES specimens(specimen_id),flexural_load_kn REAL NOT NULL CHECK(flexural_load_kn>0));
CREATE TABLE specimen_costs(cost_id INTEGER PRIMARY KEY,specimen_id TEXT NOT NULL UNIQUE REFERENCES specimens(specimen_id),steel_ratio_pct_source REAL NOT NULL,bamboo_ratio_pct_source REAL NOT NULL,cost_brl REAL NOT NULL CHECK(cost_brl>0));
CREATE TABLE source_flexural_summary(test_code TEXT NOT NULL REFERENCES test_setup(test_code),group_id TEXT NOT NULL REFERENCES reinforcement_groups(group_id),specimen_count INTEGER NOT NULL,mean_load_kn REAL NOT NULL,min_load_kn REAL NOT NULL,max_load_kn REAL NOT NULL,PRIMARY KEY(test_code,group_id));
CREATE TABLE analysis_settings(setting_id INTEGER PRIMARY KEY CHECK(setting_id=1),load_weight REAL NOT NULL CHECK(load_weight BETWEEN 0 AND 1));
CREATE TABLE mcda_weights(load_weight REAL PRIMARY KEY CHECK(load_weight BETWEEN 0 AND 1));
CREATE TABLE project_metadata(key TEXT PRIMARY KEY,value TEXT NOT NULL);
"""
views_sql="""
CREATE VIEW v_specimen_analysis AS
SELECT s.*,t.test_code,g.group_name,g.is_reference,r.flexural_load_kn,c.cost_brl,c.steel_ratio_pct_source,c.bamboo_ratio_pct_source,
       r.flexural_load_kn/c.cost_brl AS recorded_load_per_brl
FROM specimens s JOIN test_setup t ON t.test_number=s.test_number
JOIN reinforcement_groups g ON g.group_id=s.group_id
JOIN flexural_results r ON r.specimen_id=s.specimen_id
JOIN specimen_costs c ON c.specimen_id=s.specimen_id;
CREATE VIEW flexural_results_summary AS
WITH a AS (
SELECT test_code,group_id,group_name,is_reference,COUNT(*) AS n,AVG(flexural_load_kn) AS mean_load_kn,
 MIN(flexural_load_kn) AS min_load_kn,MAX(flexural_load_kn) AS max_load_kn,
 CASE WHEN COUNT(*)>1 THEN MAX(0.0,(SUM(flexural_load_kn*flexural_load_kn)-SUM(flexural_load_kn)*SUM(flexural_load_kn)/COUNT(*))/(COUNT(*)-1)) END AS sample_variance_kn2,
 AVG(cost_brl) AS mean_cost_brl,MIN(cost_brl) AS min_cost_brl,MAX(cost_brl) AS max_cost_brl
FROM v_specimen_analysis GROUP BY test_code,group_id,group_name,is_reference)
SELECT *,mean_load_kn/mean_cost_brl AS mean_load_per_mean_cost FROM a;
CREATE VIEW v_reference_comparisons AS
SELECT a.*,va.mean_load_kn AS va_mean_load_kn,vc.mean_load_kn AS vcs_mean_load_kn,
 100.0*(a.mean_load_kn/va.mean_load_kn-1) AS load_change_vs_va_pct,
 100.0*(a.mean_cost_brl/va.mean_cost_brl-1) AS cost_change_vs_va_pct,
 100.0*(a.mean_load_kn/vc.mean_load_kn-1) AS load_change_vs_vcs_pct
FROM flexural_results_summary a
LEFT JOIN flexural_results_summary va ON va.test_code=a.test_code AND va.group_name='VA'
LEFT JOIN flexural_results_summary vc ON vc.test_code=a.test_code AND vc.group_name='VCS';
CREATE VIEW v_mcda_normalized AS
WITH c AS(SELECT * FROM flexural_results_summary WHERE is_reference=0),
b AS(SELECT test_code,MIN(mean_load_kn) AS lo_load,MAX(mean_load_kn) AS hi_load,MIN(mean_cost_brl) AS lo_cost,MAX(mean_cost_brl) AS hi_cost FROM c GROUP BY test_code)
SELECT c.*,
 CASE WHEN b.hi_load=b.lo_load THEN 1.0 ELSE (c.mean_load_kn-b.lo_load)/(b.hi_load-b.lo_load) END AS normalized_load,
 CASE WHEN b.hi_cost=b.lo_cost THEN 1.0 ELSE (b.hi_cost-c.mean_cost_brl)/(b.hi_cost-b.lo_cost) END AS normalized_cost,
 CASE WHEN NOT EXISTS(SELECT 1 FROM c d WHERE d.test_code=c.test_code AND d.mean_cost_brl<=c.mean_cost_brl AND d.mean_load_kn>=c.mean_load_kn AND(d.mean_cost_brl<c.mean_cost_brl OR d.mean_load_kn>c.mean_load_kn)) THEN 1 ELSE 0 END AS pareto_efficient
FROM c JOIN b ON b.test_code=c.test_code;
CREATE VIEW v_mcda_scores AS
WITH scored AS(SELECT c.*,s.load_weight,1.0-s.load_weight AS cost_weight,
 c.normalized_load*s.load_weight+c.normalized_cost*(1.0-s.load_weight) AS score
FROM v_mcda_normalized c CROSS JOIN analysis_settings s)
SELECT *,RANK() OVER(PARTITION BY test_code ORDER BY score DESC) AS rank FROM scored;
CREATE VIEW v_mcda_sensitivity AS
WITH scored AS(SELECT c.test_code,c.group_name,w.load_weight,1.0-w.load_weight AS cost_weight,
 c.normalized_load*w.load_weight+c.normalized_cost*(1.0-w.load_weight) AS score
FROM v_mcda_normalized c CROSS JOIN mcda_weights w)
SELECT *,RANK() OVER(PARTITION BY test_code,load_weight ORDER BY score DESC) AS rank FROM scored;
CREATE VIEW v_summary_reconciliation AS
SELECT a.test_code,a.group_name,a.n,b.specimen_count,
 a.mean_load_kn-b.mean_load_kn AS mean_difference_kn,
 a.min_load_kn-b.min_load_kn AS min_difference_kn,
 a.max_load_kn-b.max_load_kn AS max_difference_kn
FROM flexural_results_summary a JOIN source_flexural_summary b ON b.test_code=a.test_code AND b.group_id=a.group_id;
"""
metadata={"project":"Experimental Data Analytics of Bamboo-Reinforced Cementitious Beams: Effects of Reinforcement Configuration on Maximum Load Capacity",
 "author":"Jeff-Mathieu Saint-Amour","source_mode":SOURCE_MODE,"source_files":", ".join(path.name for path in SOURCE_PATHS),
 "source_sha256":json.dumps({path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in SOURCE_PATHS}),
 "source_summary_available":str(SOURCE_SUMMARY_AVAILABLE),
 "observations":"39 specimens; 15 in 3PT and 24 in 4PT; no imputation or simulated loads/costs",
 "load_definition":"Recorded flexural load in kN; not flexural stress in MPa. Support span, loading geometry details, and load definition need protocol confirmation.",
 "ratios":"Percent values: 0.68 means 0.68 percent, not a fraction of 0.68. Preserve cost-source precision.",
 "cost_scope":"Reported reinforcement cost in BRL, as labeled in the supplied Power BI report. Detailed components and price date are not established. Not lifecycle cost or a market quote.",
 "species":"Dendrocalamus asper is part of the supplied project context; species and treatment are not recorded as per-specimen fields.",
 "comparison_rule":"Analyze 3PT and 4PT separately; VCS unavailable in 3PT; do not infer a missing control.",
 "statistics":"Sample SD uses n-1; descriptives only, with n=3 per available test/group. Identical geometry across specimens is retained, not treated as pairing.",
 "mcda":"Candidates exclude VA and VCS; min-max load benefit and reverse min-max reported-cost score within each test; default 50/50 is an explicit scenario, not elicited preference. Tied criteria get score 1.",
 "normalization_warning":"MCDA scores depend on candidate set and test; do not compare absolute scores across 3PT and 4PT.",
 "group_distinction":"VB-II and VBC-II remain distinct despite identical nominal ratios; treatment differences are not supplied.",
 "software":f"Python {platform.python_version()}; pandas {pd.__version__}; SQLite {sqlite3.sqlite_version}"}
db_path=OUTPUT_DIR/"bamboo_beam_analysis.sqlite"
if db_path.exists():db_path.unlink()
connection=sqlite3.connect(db_path);connection.execute("PRAGMA foreign_keys=ON")
connection.executescript(schema_sql)
tables={"test_setup":tests,"reinforcement_groups":groups,"specimens":specimens_db,
        "flexural_results":results_db,"specimen_costs":costs_db,"source_flexural_summary":summary_source_db,
        "analysis_settings":pd.DataFrame({"setting_id":[1],"load_weight":[DEFAULT_LOAD_WEIGHT]}),
        "mcda_weights":pd.DataFrame({"load_weight":WEIGHT_GRID}),
        "project_metadata":pd.DataFrame(metadata.items(),columns=["key","value"])}
for name,table in tables.items():table.to_sql(name,connection,index=False,if_exists="append")
connection.executescript(views_sql);connection.commit()
assert connection.execute("PRAGMA integrity_check").fetchone()[0]=="ok"
assert not connection.execute("PRAGMA foreign_key_check").fetchall()

## 9. Cross-tool validation

**Objective.** Check agreement between pandas, independent statistics, and SQLite.

Identifiers and counts use exact comparisons. Numerical summaries and sample variance use a tolerance of `1e−8`; MCDA scores use `1e−10`. SQLite sample variance is compared with the square of pandas sample standard deviation. Alternative load weights of 0, 0.2, and 1 are checked before restoring the 0.50 baseline.

Database integrity and foreign-key checks validate the relational output. The validation table distinguishes executed checks from the unavailable Excel-summary check in CSV mode.

In [ ]:
sql_summary=pd.read_sql_query("SELECT * FROM flexural_results_summary",connection)
cross=summary.merge(sql_summary,on=["test_code","group_id"],suffixes=("_pandas","_sql"),validate="one_to_one")
assert cross.n_pandas.eq(cross.n_sql).all()
for m in ["mean_load_kn","min_load_kn","max_load_kn","mean_cost_brl"]:
    assert np.allclose(cross[m+"_pandas"],cross[m+"_sql"],rtol=0,atol=1e-8)
assert np.allclose(cross.sample_sd_kn**2,cross.sample_variance_kn2,rtol=0,atol=1e-8)
sql_mcda=pd.read_sql_query("SELECT * FROM v_mcda_scores",connection)
scores_check=mcda.merge(sql_mcda,on=["test_code","group_id"],suffixes=("_pandas","_sql"),validate="one_to_one")
assert np.allclose(scores_check.score_pandas,scores_check.score_sql,rtol=0,atol=1e-10)
assert scores_check.rank_pandas.eq(scores_check.rank_sql).all()
assert scores_check.pareto_efficient_pandas.eq(scores_check.pareto_efficient_sql).all()
for weight in [0.0,0.2,1.0]:
    connection.execute("UPDATE analysis_settings SET load_weight=? WHERE setting_id=1",(weight,))
    alternate=pd.read_sql_query("SELECT test_code,group_name,score FROM v_mcda_scores",connection)
    expected=normalized.assign(expected=weight*normalized.normalized_load+(1-weight)*normalized.normalized_cost)
    comparison=alternate.merge(expected[["test_code","group_name","expected"]],on=["test_code","group_name"],validate="one_to_one")
    assert np.allclose(comparison.score,comparison.expected,rtol=0,atol=1e-10)
connection.execute("UPDATE analysis_settings SET load_weight=? WHERE setting_id=1",(DEFAULT_LOAD_WEIGHT,));connection.commit()
validation_report=pd.DataFrame([
 {"validation":"Source summary vs recomputed specimen statistics","passed":True if SOURCE_SUMMARY_AVAILABLE else None,"tolerance":"1e-8 kN; counts exact" if SOURCE_SUMMARY_AVAILABLE else "Not executed: original Excel summary unavailable"},
 {"validation":"Independent math/statistics vs pandas","passed":True,"tolerance":"1e-10"},
 {"validation":"SQLite vs pandas sample variance and means","passed":True,"tolerance":"1e-8"},
 {"validation":"SQLite vs pandas MCDA scores/ranks/Pareto","passed":True,"tolerance":"1e-10; ranks and flags exact"},
 {"validation":"MCDA weight changes at 0, 0.2 and 1","passed":True,"tolerance":"1e-10"},
 {"validation":"SQLite integrity and foreign keys","passed":True,"tolerance":"Exact"}])
show_table(validation_report)

## 10. Standalone SQL analysis

**Objective.** Provide both portable research queries and an independently executable database script.

`Bamboo_Beam_Analysis_Queries.sql` targets the populated SQLite database. `Bamboo_Beam_DBeaver_Complete.sql` contains the schema, observations, views, settings, and queries needed to initialize an empty SQLite database. The complete script is tested independently in memory against the notebook database.

The queries cover experimental coverage, completeness, descriptive statistics, configuration detail, Pareto trade-offs, weighted ranking, sensitivity, reference comparisons, and precision audits. Changing the stored decision weight changes the live ranking views.

In [ ]:
queries_sql="""-- BAMBOO BEAM EXPERIMENTAL ANALYSIS — ALL QUERIES USE RECORDED OBSERVATIONS
-- Q1. Experimental design and available test/group cells
SELECT test_code,group_name,COUNT(*) AS specimens FROM v_specimen_analysis GROUP BY test_code,group_name ORDER BY test_code,group_name;
-- Q2. Data completeness and valid relationships
PRAGMA integrity_check;
PRAGMA foreign_key_check;
SELECT COUNT(*) AS joined_specimens,COUNT(flexural_load_kn) AS loads,COUNT(cost_brl) AS costs FROM v_specimen_analysis;
-- Q3. Descriptive load statistics by test and reinforcement group
-- SQLite reports SAMPLE VARIANCE (kN squared); sample SD is its square root.
SELECT * FROM flexural_results_summary ORDER BY test_code,mean_load_kn DESC;
-- Q4. Recorded loads, nominal ratios, dimensions, and reported costs
SELECT * FROM v_specimen_analysis ORDER BY test_code,group_name,specimen_id;
-- Q5. Candidate cost/load trade-offs and Pareto efficiency (controls excluded)
SELECT test_code,group_name,mean_load_kn,mean_cost_brl,mean_load_per_mean_cost,pareto_efficient FROM v_mcda_normalized ORDER BY test_code,mean_cost_brl;
-- Q5b. Weighted ranking; default is 50% load and 50% cost
SELECT test_code,group_name,load_weight,cost_weight,normalized_load,normalized_cost,score,rank FROM v_mcda_scores ORDER BY test_code,rank,group_name;
-- Optional decision-weight scenario (inactive example).
-- UPDATE analysis_settings SET load_weight=0.70 WHERE setting_id=1;
-- COMMIT;
-- Baseline weight setting (inactive example):
-- UPDATE analysis_settings SET load_weight=0.50 WHERE setting_id=1;
-- COMMIT;
-- Q5c. Full 0-to-1 weight sensitivity; ties retain equal rank
SELECT * FROM v_mcda_sensitivity ORDER BY test_code,load_weight,rank,group_name;
-- Q6. Within-test comparison with VA and VCS; missing VCS in 3PT stays NULL
SELECT * FROM v_reference_comparisons ORDER BY test_code,group_name;
-- Q7. Recomputed vs workbook summary; all counts and differences should reconcile
SELECT *,CASE WHEN n=specimen_count AND ABS(mean_difference_kn)<1e-8 AND ABS(min_difference_kn)<1e-8 AND ABS(max_difference_kn)<1e-8 THEN 'OK' ELSE 'CHECK' END AS reconciliation FROM v_summary_reconciliation;
-- Q8. Detail table ready for dashboard exploration; do not pool test types
SELECT specimen_id,specimen_label,test_code,group_name,width_mm,height_mm,length_mm,steel_ratio_pct,bamboo_ratio_pct,flexural_load_kn,cost_brl,recorded_load_per_brl FROM v_specimen_analysis ORDER BY test_code,group_name,specimen_id;
-- Q9. Traceability and ratio rounding audit
SELECT specimen_id,steel_ratio_pct,steel_ratio_pct_source,steel_ratio_pct_source-steel_ratio_pct AS steel_delta_percentage_points,bamboo_ratio_pct,bamboo_ratio_pct_source FROM v_specimen_analysis ORDER BY specimen_id;
SELECT * FROM project_metadata ORDER BY key;
"""
(OUTPUT_DIR/"Bamboo_Beam_Analysis_Queries.sql").write_text(queries_sql,encoding="utf-8")
def sql_literal(value):
    if pd.isna(value):return "NULL"
    if isinstance(value,(int,float,np.integer,np.floating)):return repr(value.item() if hasattr(value,"item") else value)
    return "'"+str(value).replace("'","''")+"'"
setup=["""/* BAMBOO BEAM EXPERIMENTAL ANALYSIS — COMPLETE SQLITE SCRIPT
Author: Jeff-Mathieu Saint-Amour.
Target: empty SQLite database; compatible with DBeaver.
PART I: schema, records, and analytical views in one transaction.
PART II: independent SQL research queries.
Recorded experiments are preserved; no loads or costs are imputed or simulated.
3PT and 4PT are separate populations. Costs are reported reinforcement values in BRL; detailed composition and
price date are not established. MCDA uses an explicit 50/50 baseline, with sensitivity scenarios.
Use recorded load (kN), not stress (MPa). This analysis is not a structural design check.
Transaction failures require rollback before database reuse.
*/
PRAGMA foreign_keys=ON;
BEGIN TRANSACTION;
""",schema_sql]
for name,table in tables.items():
    if table.empty:
        continue
    setup.append("INSERT INTO "+name+" ("+", ".join(table.columns)+") VALUES\n"+",\n".join("("+", ".join(sql_literal(x) for x in row)+")" for row in table.itertuples(index=False,name=None))+";\n")
setup += [views_sql,"\nCOMMIT;\n-- PART II. VERIFICATION AND RESEARCH QUESTIONS\n",queries_sql]
complete_sql="\n".join(setup)
(OUTPUT_DIR/"Bamboo_Beam_DBeaver_Complete.sql").write_text(complete_sql,encoding="utf-8")
independent_db=sqlite3.connect(":memory:");independent_db.executescript(complete_sql)
assert independent_db.execute("SELECT COUNT(*) FROM specimens").fetchone()[0]==39
assert independent_db.execute("PRAGMA integrity_check").fetchone()[0]=="ok"
assert not independent_db.execute("PRAGMA foreign_key_check").fetchall()
assert independent_db.execute("SELECT * FROM v_mcda_scores ORDER BY test_code,group_id").fetchall()==connection.execute("SELECT * FROM v_mcda_scores ORDER BY test_code,group_id").fetchall()
independent_db.close();print("Standalone DBeaver script tested successfully on an empty SQLite database.")

## 11. CSV exports and research findings

**Objective.** Deliver model-ready data and a calculated synthesis of the experiment.

Exports retain stable specimen and group keys, source numeric precision, explicit units, audit tables, and validation results. The principal Power BI interface comprises `fact_beam_analysis.csv`, `test_setup.csv`, and `reinforcement_groups.csv`. Dimension-to-fact relationships use `test_number` and `group_id`.

The generated findings distinguish mean-load leaders, raw load-to-cost ratios, and the 50/50 scenario. Conclusions are limited to the supplied experiment and its reported reinforcement cost basis. Price date, detailed cost composition, per-specimen treatment, support span, and complete loading protocol are not established by the analytical tables.

In [ ]:
exports={**tables,"fact_beam_analysis":joined,"group_statistics":summary,"reference_comparisons":reference_comparisons,
 "mcda_candidates":mcda,"mcda_sensitivity":sensitivity,"quality_checks":pd.DataFrame(checks),
 "ratio_precision_audit":ratio_audit,"source_summary_reconciliation":summary_reconciliation.drop(columns="_merge"),
 "independent_statistics_check":independent_check,"geometry_reuse_audit":geometry_reuse,"validation_report":validation_report}
for name,table in exports.items():table.to_csv(CSV_DIR/f"{name}.csv",index=False,encoding="utf-8-sig",float_format="%.12g")
(OUTPUT_DIR/"project_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
leaders=summary.loc[summary.groupby("test_code").mean_load_kn.idxmax()]
recommendations=mcda[mcda["rank"].eq(1)]
finding_lines=["# Validated descriptive findings", "", "All values below come from recorded specimen data; 3PT and 4PT remain separate.", ""]
for _,row in leaders.iterrows():finding_lines.append(f"- {row.test_code}: the highest recorded group mean is {row.group_name}, {row.mean_load_kn:.4f} kN (n={int(row.n)}, sample SD={row.sample_sd_kn:.4f} kN).")
for _,row in recommendations.iterrows():finding_lines.append(f"- {row.test_code}: {row.group_name} ranks first in the explicitly assumed 50/50 candidate-only load/cost scenario (score={row.score:.4f}); this is not a universal optimum.")
finding_lines += ["", "## Interpretation limits", "- VCS is unavailable in 3PT; its comparative result stays missing.",
 "- VB-II and VBC-II remain separate; nominal ratios alone do not identify the omitted treatment details.",
 "- The precise steel percentages in costs reconcile with the rounded nominal specimen percentages.",
 "- Recorded loads are not converted into MPa: support span and protocol definitions are not supplied.",
 "- Three specimens per test/group support descriptive comparisons, not broad claims of statistical superiority.",
 "- Cost is the reported reinforcement figure in BRL; detailed composition and price date need confirmation.",
 "- No lifecycle, environmental impact, field durability, code compliance, or structural design suitability is established here."]
(OUTPUT_DIR/"Validated_Findings.md").write_text("\n".join(finding_lines),encoding="utf-8")
print("\n".join(finding_lines));connection.close()
print(f"Exported {len(exports)} CSV tables, 4 figures, SQLite, and SQL scripts.")

## 12. Output packaging

**Objective.** Consolidate reproducible analytical deliverables.

The results archive contains SQLite, SQL scripts, CSV tables, four figures, metadata, and calculated findings. The notebook supports Colab downloads and local output folders. The repository documents Excel and CSV input routes; both require separately authorized research inputs.

Results describe recorded load and reported reinforcement cost. They do not establish code compliance, structural design suitability, lifecycle cost, environmental impact, or field durability.

In [ ]:
RESULT_ZIP=OUTPUT_DIR.parent/"Bamboo_Beam_Results.zip"
with zipfile.ZipFile(RESULT_ZIP,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():archive.write(path,path.relative_to(OUTPUT_DIR))
print("Results:",RESULT_ZIP.name)
try:
    from google.colab import files
except ImportError:
    print("Local run complete; open the output folder or results ZIP.")
else:
    files.download(str(RESULT_ZIP))

## 13. Scientific discussion and conclusions

The experiment supports differentiated conclusions by test setup and decision criterion. In 3PT, VAB-III combines the highest recorded group mean with the highest bamboo-containing load-to-cost ratio. Relative to VA, its mean load is 43.17% higher and its reported reinforcement cost is 36.55% higher, corresponding to a 4.85% ratio improvement. The equal-weight normalized model instead favors VAB-II, illustrating how explicit preferences can change candidate selection.

In 4PT, VA leads both overall mean load and overall load-to-cost ratio. Among bamboo-containing groups, VB-III leads mean load and VBC-II leads the ratio. VBC-II costs 11.01% more than VA while recording an 8.10% lower mean load. Candidate-only leadership therefore does not establish an advantage over the reference configuration.

Variability adds a separate dimension to interpretation. The lowest 4PT CV belongs to VAB-III (1.72%), whereas VB-IV has the highest (22.07%). Low dispersion does not itself imply high load capacity, and a large individual load does not establish a superior group mean.

### Scope of inference

Three specimens per available combination provide limited information about the underlying load distribution. No hypothesis testing, population-level superiority, or causal mechanism is established. Shared nominal ratios and repeated dimensions do not establish matched specimens or equivalent treatment. Per-specimen bamboo species and treatment, support span, full loading protocol, detailed cost composition, and price date are not recorded in the analytical tables.

The findings support further experimental assessment and transparent comparison of recorded configurations. They do not establish code compliance, structural design suitability, field durability, lifecycle cost, or environmental performance.